In [7]:
import numpy as np
import matplotlib.pyplot as plt

print("NumPy импортирован успешно")

NumPy импортирован успешно


In [8]:
states = ['Солнечно', 'Дождливо']

P = np.array([
    [0.8, 0.2],  # из Солнечно: 0.8 остаться, 0.2 в Дождливо
    [0.4, 0.6],  # из Дождливо: 0.4 в Солнечно, 0.6 остаться
])

print("Матрица переходов P:")
print(P)

row_sums = np.sum(P, axis=1)  # сумма по строкам (axis=1)

print("Суммы строк (должны быть равны 1):", row_sums)

Матрица переходов P:
[[0.8 0.2]
 [0.4 0.6]]
Суммы строк (должны быть равны 1): [1. 1.]


**Вопрос для самопроверки:** Почему сумма элементов каждой строки обязательно должна быть равна единице? Свяжите ответ с понятием распределения вероятностей из лекции 1.

**Ответ:** Строка матрицы переходов `P[i]` задаёт условное распределение вероятностей следующего состояния при условии, что текущее состояние равно `i`. Поскольку следующее состояние обязано быть каким-то из всех возможных состояний (события «перейти в состояние j» образуют полную группу несовместных событий), сумма вероятностей всех исходов должна равняться 1 — это аксиома нормировки распределения вероятностей. Иначе матрица не была бы стохастической, и величины `P[i][j]` нельзя было бы интерпретировать как вероятности.

In [9]:
np.random.seed(42)

n_steps = 15
current_state = 0  # начинаем с "Солнечно"
sequence = [current_state]

for step in range(n_steps - 1):
    # Выбираем следующее состояние согласно строке текущего состояния матрицы P
    next_state = np.random.choice([0, 1], p=P[current_state])
    sequence.append(next_state)
    current_state = next_state

# Переводим индексы в названия состояний
sequence_names = [states[s] for s in sequence]

print("Траектория цепи Маркова:")
print(" → ".join(sequence_names))

Траектория цепи Маркова:
Солнечно → Солнечно → Дождливо → Дождливо → Дождливо → Солнечно → Солнечно → Солнечно → Дождливо → Дождливо → Дождливо → Солнечно → Дождливо → Дождливо → Солнечно


**Вопрос для самопроверки:** Запустите симуляцию ещё раз БЕЗ повторной установки `np.random.seed(42)` (то есть выполните ячейку с циклом ещё раз, не выполняя ячейку с seed). Получится ли та же самая траектория? Почему?

**Ответ:** Нет, траектория будет другой. `np.random.seed(42)` задаёт начальное состояние генератора псевдослучайных чисел, обеспечивая воспроизводимость. Если повторно выполнить цикл без переустановки seed, генератор продолжит работу с того места, где остановился, и выдаст следующую порцию «случайных» чисел — соответственно, выборы состояний будут другими. Чтобы воспроизвести ту же самую траекторию, нужно каждый раз заново выполнять `np.random.seed(42)` перед циклом.

In [13]:
P2 = P @ P  # умножение матриц в NumPy (оператор @)
print("P^2 =")
print(P2)

P10 = np.linalg.matrix_power(P, 10)  # возведение матрицы в степень
print("P^10 =")
print(P10)

# Вероятность оказаться в "Дождливо" через 2 шага, начиная с "Солнечно"
prob_rainy_in_2_steps = P2[0, 1]  # строка 0 = Солнечно, столбец 1 = Дождливо

print(f"P(Дождливо через 2 шага | Солнечно сегодня) = {prob_rainy_in_2_steps:.4f}")

P^2 =
[[0.72 0.28]
 [0.56 0.44]]
P^10 =
[[0.66670162 0.33329838]
 [0.66659676 0.33340324]]
P(Дождливо через 2 шага | Солнечно сегодня) = 0.2800


**Вопрос для самопроверки:** Сравните первую и вторую строки матрицы P10. Насколько они похожи друг на друга? Что это означает с точки зрения влияния начального состояния на распределение вероятностей через много шагов?

**Ответ:** Первая и вторая строки `P^10` практически совпадают (с точностью до ~1e-3–1e-4). Это означает, что при возведении матрицы в большую степень влияние начального состояния «размывается»: распределение вероятностей через 10 шагов почти не зависит от того, откуда мы стартовали. Строки матрицы сходятся к стационарному распределению π ≈ [0.6667, 0.3333]. Такое поведение характерно для регулярных (эргодических) цепей Маркова: независимо от начального состояния распределение сходится к единственному стационарному распределению. Это прямая подготовка к теме лекции 6.

In [15]:
np.random.seed(42)

n_steps = 10000
current_state = 0
sequence = [current_state]

for step in range(n_steps - 1):
    current_state = np.random.choice([0, 1], p=P[current_state])
    sequence.append(current_state)

sequence = np.array(sequence)

freq_sunny = np.mean(sequence == 0)  # доля шагов в состоянии "Солнечно"
freq_rainy = np.mean(sequence == 1)

print(f"Эмпирическая доля 'Солнечно': {freq_sunny:.4f}")
print(f"Эмпирическая доля 'Дождливо': {freq_rainy:.4f}")
print("Теоретическое значение (из P^10): [0.6667, 0.3333]")

Эмпирическая доля 'Солнечно': 0.6783
Эмпирическая доля 'Дождливо': 0.3217
Теоретическое значение (из P^10): [0.6667, 0.3333]


**Вопрос для самопроверки:** Насколько близки эмпирические доли к теоретическим значениям? Какому понятию из Практики 1 (первая неделя курса) соответствует это наблюдение?

**Ответ:** Эмпирические доли очень близки к теоретическим (обычно совпадение до 2–3 знаков после запятой: ~0.666x и ~0.333x). Это иллюстрирует **закон больших чисел**: при увеличении числа независимых испытаний (шагов цепи) эмпирическая частота события сходится по вероятности к его теоретической вероятности. В Практике 1 это понятие вводилось при обсуждении частотной интерпретации вероятности и сходимости выборочных средних. Здесь мы наблюдаем ту же идею: средняя доля времени, проведённого в состоянии, сходится к стационарной вероятности этого состояния.

In [16]:
my_states = ['Главная', 'Каталог', 'Корзина']

# Своя матрица переходов 3x3 (строки должны суммироваться в 1!)
my_P = np.array([
    [0.5, 0.4, 0.1],
    [0.3, 0.5, 0.2],
    [0.6, 0.1, 0.3],
])

print("Суммы строк:", np.sum(my_P, axis=1))  # проверка корректности

np.random.seed(1)
current = 0
my_sequence = [current]

for step in range(14):
    current = np.random.choice([0, 1, 2], p=my_P[current])
    my_sequence.append(current)

print(" → ".join([my_states[s] for s in my_sequence]))

Суммы строк: [1. 1. 1.]
Главная → Главная → Каталог → Главная → Главная → Главная → Главная → Главная → Главная → Главная → Каталог → Каталог → Каталог → Главная → Каталог


**Задание для размышления:** Попробуйте изменить вероятности в `my_P` так, чтобы состояние «Корзина» встречалось в траектории значительно чаще. Какие элементы матрицы нужно увеличить?

**Ответ:** Чтобы «Корзина» (индекс 2) встречалась чаще, нужно:
1. Увеличить вероятности **перехода в** состояние 2 из других состояний, т.е. элементы **третьего столбца**: `my_P[0, 2]`, `my_P[1, 2]` (и, соответственно, уменьшить другие элементы этих строк, чтобы сумма осталась 1).
2. Увеличить вероятность **остаться** в «Корзине», т.е. `my_P[2, 2]` (диагональный элемент третьей строки) — тогда, попав в корзину, пользователь будет дольше в ней задерживаться.

Например: `my_P = [[0.3, 0.3, 0.4], [0.2, 0.3, 0.5], [0.1, 0.1, 0.8]]` — тогда стационарная вероятность «Корзины» заметно возрастёт.